# 02 - Realtime Tracking

Este notebook orquestra a engine de captura de pose em tempo real. Toda a logica reutilizavel fica em `core/`; o notebook apenas conecta os componentes e executa o loop da webcam.

In [1]:
from __future__ import annotations

import time
from pathlib import Path

import cv2
import numpy as np

from core.camera_capture import CameraCapture
from core.joint_angle_calculator import JointAngleCalculator
from core.pose_detector import PoseDetector
from core.pose_frame import PoseFrame
from core.pose_normalizer import PoseNormalizer
from core.pose_source import RealtimePoseSource
from core.visualizers.debug_panel import DebugPanel
from core.visualizers.realtime_view import RealtimeView

## Arquitetura

A normalizacao usa o centro do quadril como origem e a distancia entre o centro dos ombros e o centro dos quadris como escala. Essa escolha reduz diferencas de altura, distancia da camera, posicao no quadro e proporcoes corporais, mantendo a estrutura pronta para uma futura etapa de correcao de rotacao.

In [2]:
class RealTimePoseTracker:
    """Coordinate real-time pose tracking components from core."""

    def __init__(self, camera_index: int = 0, model_path: str | Path | None = None) -> None:
        self.camera_index = camera_index
        self.model_path = model_path
        self.camera: CameraCapture | None = None
        self.detector: PoseDetector | None = None
        self.normalizer = PoseNormalizer()
        self.angle_calculator = JointAngleCalculator()
        self.pose_source: RealtimePoseSource | None = None
        self.realtime_view = RealtimeView()
        self.debug_panel = DebugPanel()
        self.fps = 0.0
        self._last_tick = time.perf_counter()

    def initialize_camera(self) -> None:
        """Initialize webcam capture."""
        self.camera = CameraCapture(camera_index=self.camera_index)
        self.camera.open()

    def initialize_detector(self) -> None:
        """Initialize MediaPipe PoseLandmarker detector."""
        self.detector = PoseDetector(model_path=self.model_path)
        self.detector.initialize()

    def process_frame(self) -> tuple[np.ndarray, PoseFrame]:
        """Read the next webcam frame and convert it into a PoseFrame."""
        if self.pose_source is None:
            raise RuntimeError("Pose source is not initialized.")

        pose_frame = self.create_pose_frame()
        frame = self.pose_source.latest_frame
        if frame is None:
            raise RuntimeError("No webcam frame available after pose processing.")
        return frame, pose_frame

    def create_pose_frame(self) -> PoseFrame:
        """Delegate PoseFrame creation to the configured PoseSource."""
        if self.pose_source is None:
            raise RuntimeError("Pose source is not initialized.")
        return self.pose_source.get_next_pose()

    def draw_pose(self, frame: np.ndarray, pose_frame: PoseFrame) -> None:
        """Render the real-time webcam view."""
        rendered = self.realtime_view.draw(frame, pose_frame, self.fps)
        self.realtime_view.show(rendered)

    def draw_debug_panel(self, pose_frame: PoseFrame) -> None:
        """Render the technical debug panel."""
        if self.pose_source is None:
            raise RuntimeError("Pose source is not initialized.")

        normalization = self.pose_source.latest_normalization
        panel = self.debug_panel.draw(
            pose_frame=pose_frame,
            fps=self.fps,
            hip_center=normalization.hip_center,
            scale_factor=normalization.scale_factor,
        )
        self.debug_panel.show(panel)

    def run(self) -> None:
        """Start the webcam tracking loop until Q is pressed."""
        self.initialize_camera()
        self.initialize_detector()

        if self.camera is None or self.detector is None:
            raise RuntimeError("Tracker components were not initialized correctly.")

        self.pose_source = RealtimePoseSource(
            camera=self.camera,
            detector=self.detector,
            normalizer=self.normalizer,
            angle_calculator=self.angle_calculator,
        )

        try:
            while True:
                frame, pose_frame = self.process_frame()
                self._update_fps()
                self.draw_pose(frame, pose_frame)
                self.draw_debug_panel(pose_frame)

                key = cv2.waitKey(1) & 0xFF
                if key == ord("q") or key == ord("Q"):
                    break
        finally:
            self._release_resources()

    def _update_fps(self) -> None:
        """Update displayed FPS using elapsed wall time."""
        current_tick = time.perf_counter()
        elapsed = current_tick - self._last_tick
        if elapsed > 0:
            current_fps = 1.0 / elapsed
            self.fps = current_fps if self.fps == 0.0 else (self.fps * 0.9) + (current_fps * 0.1)
        self._last_tick = current_tick

    def _release_resources(self) -> None:
        """Release webcam, MediaPipe and OpenCV windows."""
        if self.camera is not None:
            self.camera.release()
        if self.detector is not None:
            self.detector.close()
        cv2.destroyAllWindows()

In [3]:
tracker = RealTimePoseTracker()
tracker.run()

KeyboardInterrupt: 